# Clarity Analytics Center — Pipeline Runner

Runs the full pipeline end to end. Run all cells in order.

The first cell clones the repo and moves into it **only if it isn't already there** —
this is a no-op when running from a local clone, and handles Colab, which starts in
`/content` even when the notebook lives inside the repository.

The remaining cells execute each pipeline notebook in place, so each one is saved with
its outputs:

1. `01_storage.ipynb` — creates the SQLite database and the Bronze/Silver/Gold schema
2. `02_ingestion.ipynb` — lands the four raw sources into Bronze
3. `03_processing.ipynb` — transforms Bronze → Silver → Gold (star schema)

Result: `storage/clarity_analytics_center.db`.

In [ ]:
import os
import sys
import sqlite3
from pathlib import Path

! pip install tabulate
from tabulate import tabulate

# check if script is being started in repo, if not clone repo and run code
if not os.path.exists("notebooks"):
    !git clone https://github.com/clopezfranco27/Group_4_Clarity_Analytics_Center.git
    %cd Group_4_Clarity_Analytics_Center

In [ ]:
!jupyter nbconvert --to notebook --execute --inplace --ExecutePreprocessor.timeout=1800 notebooks/01_storage.ipynb

In [ ]:
!jupyter nbconvert --to notebook --execute --inplace --ExecutePreprocessor.timeout=1800 notebooks/02_ingestion.ipynb

In [ ]:
!jupyter nbconvert --to notebook --execute --inplace --ExecutePreprocessor.timeout=1800 notebooks/03_processing.ipynb

In [ ]:
# display tables and columns in database

db_path = 'storage/clarity_analytics_center.db'

if not Path(db_path).exists():
    print('database file [{db_path}] doesn\'t exist')
    sys.exit(1)

    
conn = sqlite3.connect(db_path)

cursor = conn.cursor()
cursor.execute(f'SELECT name FROM sqlite_master WHERE type="table" '
                f'AND name != "sqlite_sequence"')
tables = cursor.fetchall()

table_data = []
for table_name_tuple in tables:
    table_name = table_name_tuple[0]

    # get columns
    cursor.execute(f"PRAGMA table_info('{table_name}');")
    columns = cursor.fetchall()
    num_columns = len(columns)

    # get rows
    cursor.execute(f"SELECT COUNT(*) FROM '{table_name}';")
    num_rows = cursor.fetchone()[0]

    table_data.append([table_name, num_columns, num_rows])

table_data.sort(key=lambda x: x[0])
print("Tables in the database:")
print(tabulate(table_data,
        headers=["Table Name", "Number of Columns", "Number of Rows"],
        tablefmt="fancy_grid"))

In [ ]:
conn.close()